# EchoGesture — College ML Showcase
Run the cells in order to inspect the models and recognise uploaded signs.

| Task | Classifier | Input |
|---|---|---|
| 24 static letters | Geometric hand features + SVM | Image |
| J and Z | TemporalMatcher | Video |
| 81 ASL words | Hand/body sequence features + SVM | Video |

Select **Runtime → Change runtime type → Runtime Version → 2026.07** for Python 3.12. The project uses MediaPipe 0.10.21 and its Solutions API. CPU is sufficient.

Files uploaded here are processed in Google's hosted runtime. Use recordings you have permission to upload. Colab provides the college demonstration; edge latency requires measurement on the target device.


In [ ]:
import sys
if sys.version_info[:2] not in [(3,10),(3,11),(3,12)]:
    raise RuntimeError('Choose Colab runtime 2026.07 (Python 3.12), then rerun.')
print(sys.version)


## Load the repository
For the private repository, enter a GitHub token with read access into the hidden prompt. The token is used only for cloning. The notebook contains no saved credentials. You may also run from an existing local repository checkout.


In [ ]:
import os, subprocess, tempfile
from pathlib import Path
from getpass import getpass
repo = Path('/content/EchoGesture') if Path('/content').exists() else Path.cwd()
if not (repo / 'app.py').exists():
    url = 'https://github.com/MTJ19/EchoGesture.git'
    attempt = subprocess.run(['git','clone','--depth','1',url,str(repo)], capture_output=True,
                             env={**os.environ,'GIT_TERMINAL_PROMPT':'0'})
    if attempt.returncode:
        token = getpass('GitHub repository read token: ')
        try:
            with tempfile.TemporaryDirectory() as directory:
                helper = Path(directory) / 'askpass'
                helper.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo x-access-token ;;\n*) printf "%s" "$ECHO_GITHUB_TOKEN" ;;\nesac\n')
                helper.chmod(0o700)
                env = {**os.environ,'GIT_ASKPASS':str(helper),'GIT_TERMINAL_PROMPT':'0','ECHO_GITHUB_TOKEN':token}
                result = subprocess.run(['git','clone','--depth','1',url,str(repo)],env=env,capture_output=True)
                if result.returncode:
                    raise RuntimeError('Clone failed. Check token and repository access.')
        finally:
            token = None
            env = None
os.chdir(repo)
print('Repository loaded')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print('Dependencies installed. If a restart is requested, restart the session and rerun the cells.')


In [ ]:
import json, joblib
from app import create_app
client = create_app().test_client()
alphabet = client.get('/api/status').get_json()
words = client.get('/api/words/status').get_json()
assert alphabet['trained'] and words['trained']
print('Static alphabet:', ' '.join(alphabet['labels']))
print('Word vocabulary:',len(words['words']))
print(', '.join(words['words']))
motion = joblib.load('models/motion.joblib')
print('Motion classifier:',motion['model'])
print('Motion training examples:',motion['training_count'])


## Recognise an uploaded sign
Choose letter for an image, motion for a complete J/Z video, or word for a supported ASL word. Use one complete sign per video, up to 10 seconds and 8 MB. Model scores can be high on incorrect predictions.


In [ ]:
from io import BytesIO
from PIL import Image
import cv2
from asl_lab.features import HandExtractor

def recognise(path, mode):
    path = Path(path)
    if mode == 'letter':
        image = Image.open(path).convert('RGB')
        buffer = BytesIO()
        image.save(buffer,format='PNG')
        buffer.seek(0)
        response = client.post('/api/predict',data={'image':(buffer,'image.png'),'cutoff':'30'})
    elif mode == 'word':
        with path.open('rb') as stream:
            response = client.post('/api/words/predict',data={'video':(stream,path.name)})
    elif mode == 'motion':
        tracker = HandExtractor(static_image_mode=False)
        cap = cv2.VideoCapture(str(path))
        frames = []
        try:
            fps = cap.get(cv2.CAP_PROP_FPS)
            if not 0 < fps <= 120:
                raise ValueError('Invalid frame rate')
            index, next_ms = 0, 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                timestamp = index*1000/fps
                if timestamp > 10000:
                    raise ValueError('Video exceeds 10 seconds')
                if timestamp >= next_ms:
                    features, status = tracker.extract(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
                    if features is not None:
                        frames.append(dict(points=tracker.points,time=timestamp,aspect=frame.shape[1]/frame.shape[0]))
                    next_ms += 100
                index += 1
        finally:
            cap.release()
            tracker.close()
        response = client.post('/api/motion',json={'action':'predict','frames':frames})
    else:
        raise ValueError('Choose letter, word or motion')
    result = response.get_json()
    if response.status_code != 200:
        raise ValueError((result or {}).get('error','Prediction failed'))
    return {k:v for k,v in result.items() if k not in ('landmarks','supported_words')}


In [ ]:
#@title Upload and predict
mode = 'word' #@param ['letter','motion','word']
from google.colab import files
from IPython.display import display, Image as DisplayImage, Video
uploaded = files.upload()
with tempfile.TemporaryDirectory(prefix='echogesture-colab-') as directory:
    for filename, content in uploaded.items():
        path = Path(directory)/Path(filename).name
        path.write_bytes(content)
        if mode == 'letter':
            display(DisplayImage(data=content))
        else:
            display(Video(data=content,embed=True,mimetype='video/mp4'))
        try:
            print(filename, json.dumps(recognise(path,mode),indent=2))
        except Exception as error:
            print(filename,str(error))
uploaded.clear()


## Inspect the ML models and saved results
Training used WLASL and matching ASL Citizen signs for words, and SignAlphaSet for J/Z. The following cell loads fitted models and recorded reports. The reports describe earlier experiments. Repeatedly inspected clips are regression checks.


In [ ]:
print('Alphabet:',joblib.load('models/classifier.joblib')['model'])
print('Words:',joblib.load('models/word81-grammar-candidate.joblib'))
print('Word model size in MB:',round(Path('models/word81-grammar-candidate.joblib').stat().st_size/1e6,2))
for filename in ['word81-grammar.json','word81-grammar-uploads.json','word81-broad-heldout-comparison.json']:
    print(filename)
    display(json.loads((Path('reports')/filename).read_text()))


## Evaluate new labelled files
Name recordings APPLE__01.mp4 or J__01.mp4, with the expected label before the double underscore. Choose one task per batch. Keep evaluation signers and recordings separate from training. Errors count as misses. Timing includes decoding and tracking on Colab.


In [ ]:
import time,csv
evaluation_mode = 'word' #@param ['letter','motion','word']
uploaded = files.upload()
trials = []
with tempfile.TemporaryDirectory() as directory:
    for filename, content in uploaded.items():
        path = Path(directory)/Path(filename).name
        path.write_bytes(content)
        expected = path.stem.split('__')[0].upper()
        started = time.perf_counter()
        try:
            result = recognise(path,evaluation_mode)
            predicted = result.get('label')
            predicted = predicted.upper() if predicted else None
            error = ''
        except Exception as exc:
            predicted,error = None,str(exc)
        trials.append(dict(file=path.name,expected=expected,predicted=predicted,correct=predicted==expected,
                           request_ms=round((time.perf_counter()-started)*1000,1),error=error))
uploaded.clear()
display(trials)
if trials:
    print('Correct:',sum(r['correct'] for r in trials),'/',len(trials))
    report = Path('/content/echogesture_results.csv')
    with report.open('w',newline='') as stream:
        writer = csv.DictWriter(stream,fieldnames=list(trials[0]))
        writer.writeheader()
        writer.writerows(trials)
    files.download(str(report))


## Scope and references
This notebook demonstrates existing trained models. Full retraining requires separately downloaded datasets and prepared features described in the repository. Raw datasets are excluded from GitHub.

The active vocabulary contains 81 words. J/Z uses an experimental model with limited independent live validation. Similar signs, different signers, missing landmarks and lighting affect results. Quantisation and target-device latency measurement are pending for the hackathon.

[WLASL](https://dxli94.github.io/WLASL/) · [ASL Citizen](https://www.microsoft.com/en-us/research/project/asl-citizen/) · [SignAlphaSet v3](https://doi.org/10.17632/8fmvr9m98w.3) · [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)
